# IOAI — 2025 Team Selection Day2 Ml Player Clustering (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-team-selection-day2-ml-player-clustering'
for f in ['train.csv','sample_submission.csv']:
    if not os.path.exists(f): os.system(f'wget -q {BASE}/{f}')
print('데이터 준비:', [f for f in ['train.csv','sample_submission.csv'] if os.path.exists(f)])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Day 2 — Player Clustering · 모범답안 (원본 대회 코드)

> **Kazakhstan – Team Selection 2025 · Day 2 (Unsupervised)**

이 노트북은 Batyr Yerdenov 의 **실제 TST 제출 코드**(github.com/batyrq/ioai-tst-kazakhstan)를 거의 **그대로** 옮긴 것입니다.
데이터는 공개 up-solving 미러(`tst-day-2-upsolving`)의 원본을 동봉본으로 사용하며, 바뀐 부분은 **데이터 경로**와 **로컬 채점용 검증 예측 추가**뿐입니다(주석으로 표시). 더 정돈·향상된 재구현은 **모범답안2** 탭을 참고하세요.


In [ ]:
# --- 데이터 경로 어댑터(추가): 동봉 데이터 폴더로 이동 (원본 코드는 그대로) ---
# 현재 폴더(".")를 먼저 확인 → 모범답안 작업본(_solution)에 머물러 제출물이 거기에 저장되게 한다
# (상위 폴더에도 데이터가 풀려 있어 "."보다 ".."를 먼저 보면 self(내 풀이) 폴더에 잘못 저장됨).
import os
for _b in [".", "..", "../.."]:
    if os.path.exists(os.path.join(_b, "train.csv")):
        os.chdir(_b); break
print("data dir:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# === 1. Загрузка данных ===
df = pd.read_csv("train.csv")
sample = pd.read_csv("sample_submission.csv")

# === 2. Мета-признаки ===
df["attacking_skill"] = df[["finishing", "positioning", "shot_power", "volleys", "long_shots"]].mean(axis=1)
df["passing_ability"] = df[["short_passing", "long_passing", "vision", "crossing"]].mean(axis=1)
df["dribble_mobility"] = df[["dribbling", "agility", "balance", "ball_control"]].mean(axis=1)
df["pace"] = df[["acceleration", "sprint_speed"]].mean(axis=1)
df["defense_skill"] = df[["interceptions", "standing_tackle", "sliding_tackle", "defensive_awareness"]].mean(axis=1)
df["physicality"] = df[["strength", "stamina", "jumping", "aggression"]].mean(axis=1)
df["set_piece_specialist"] = df[["curve", "fk_accuracy", "penalties"]].mean(axis=1)
df["goalkeeper_score"] = df[["gk_diving", "gk_handling", "gk_kicking", "gk_positioning", "gk_reflexes"]].mean(axis=1)
df["composure_score"] = df[["composure", "reactions"]].mean(axis=1)
df["offensive_support"] = df[["vision", "positioning", "short_passing"]].mean(axis=1)

# Новые признаки
df["attack_support"] = df[["vision", "short_passing", "composure"]].mean(axis=1)
df["defending_positioning"] = df[["defensive_awareness", "positioning", "aggression"]].mean(axis=1)

# === 3. Вратари ===
gk_cols = ["gk_diving", "gk_handling", "gk_kicking", "gk_positioning", "gk_reflexes"]
df["is_gk"] = df[gk_cols].gt(40).all(axis=1)

# === 4. Деление ===
gk_df = df[df["is_gk"]].copy()
field_df = df[~df["is_gk"]].copy()

# === 5. Признаки ===
features = [
    "attacking_skill", "passing_ability", "dribble_mobility", "pace",
    "defense_skill", "physicality", "set_piece_specialist",
    "composure_score", "offensive_support", "attack_support", "defending_positioning"
]
gk_features = ["goalkeeper_score"]

def preprocess(X):
    X = SimpleImputer(strategy="mean").fit_transform(X)
    X = StandardScaler().fit_transform(X)
    return X

X_field = preprocess(field_df[features])
X_gk = preprocess(gk_df[gk_features])

# === 6. Оптимальный n_clusters для field игроков (по BIC или силуэту) ===
best_score = -1
best_k = 6
for k in range(5, 15):
    gmm = GaussianMixture(n_components=k, random_state=42)
    labels = gmm.fit_predict(X_field)
    score = silhouette_score(X_field, labels)
    if score > best_score:
        best_score = score
        best_k = k

print(f"Лучшее число кластеров: {best_k}")

# === 7. Кластеризация ===
gmm_field = GaussianMixture(n_components=best_k, random_state=42).fit(X_field)
gmm_gk = GaussianMixture(n_components=1, random_state=42).fit(X_gk)

field_clusters = gmm_field.predict(X_field)
gk_clusters = gmm_gk.predict(X_gk)

field_df["cluster"] = field_clusters
gk_df["cluster"] = gk_clusters + best_k  # отдельно

# === 8. Объединение ===
df_out = pd.concat([field_df, gk_df], axis=0)
df_out = df_out[["id", "cluster"]].sort_values("id")

# === 9. Финальный submission ===
submission = sample.drop(columns=["cluster"], errors="ignore").merge(df_out, on="id", how="left")
submission.to_csv("submission.csv", index=False)


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)